In [ ]:
!pip install yfinance plotly numpy pandas requests beautifulsoup4 duckduckgo-search --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 18.4 MB/s eta 0:00:00


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

try:
    from duckduckgo_search import DDGS
    DDGS_AVAILABLE = True
except:
    DDGS_AVAILABLE = False
    print("⚠️ duckduckgo_search no disponible, usando datos alternativos")

# ============================================================
# FUNCIONES DE INDICADORES TÉCNICOS
# ============================================================

def calcular_rsi(serie, periodo=14):
    delta      = serie.diff()
    ganancia   = delta.where(delta > 0, 0)
    perdida    = -delta.where(delta < 0, 0)
    avg_gan    = ganancia.rolling(window=periodo).mean()
    avg_per    = perdida.rolling(window=periodo).mean()
    rs         = avg_gan / avg_per
    return 100 - (100 / (1 + rs))

def calcular_macd(serie, rapida=12, lenta=26, señal=9):
    ema_r      = serie.ewm(span=rapida, adjust=False).mean()
    ema_l      = serie.ewm(span=lenta,  adjust=False).mean()
    macd       = ema_r - ema_l
    macd_sig   = macd.ewm(span=señal, adjust=False).mean()
    return macd, macd_sig, macd - macd_sig

def calcular_bollinger(serie, periodo=20, desv=2):
    sma        = serie.rolling(window=periodo).mean()
    std        = serie.rolling(window=periodo).std()
    return sma + (std * desv), sma, sma - (std * desv)

def calcular_atr(high, low, close, periodo=14):
    tr         = pd.concat([
                    high - low,
                    abs(high - close.shift()),
                    abs(low  - close.shift())
                 ], axis=1).max(axis=1)
    return tr.rolling(window=periodo).mean()

# ============================================================
# FUNCIONES DE BÚSQUEDA EN INTERNET
# ============================================================

def buscar_noticias(query, max_results=5):
    """Busca noticias usando DuckDuckGo"""
    resultados = []
    try:
        if DDGS_AVAILABLE:
            with DDGS() as ddgs:
                for r in ddgs.news(query, max_results=max_results):
                    resultados.append({
                        'titulo': r.get('title', ''),
                        'fuente': r.get('source', ''),
                        'fecha':  r.get('date', ''),
                        'url':    r.get('url', '')
                    })
    except Exception as e:
        resultados = []
    return resultados

def buscar_texto(query, max_results=3):
    """Busca texto general usando DuckDuckGo"""
    resultados = []
    try:
        if DDGS_AVAILABLE:
            with DDGS() as ddgs:
                for r in ddgs.text(query, max_results=max_results):
                    resultados.append(r.get('body', ''))
    except:
        resultados = []
    return resultados

# ============================================================
# ANÁLISIS DE SENTIMIENTO SIMPLE
# ============================================================

def analizar_sentimiento_texto(textos):
    """
    Analiza sentimiento basado en palabras clave
    Retorna: score, descripción, señales encontradas
    """
    palabras_positivas = [
        'buy', 'bullish', 'growth', 'strong', 'beat', 'surge',
        'rally', 'gain', 'profit', 'record', 'upgrade', 'outperform',
        'compra', 'alcista', 'crecimiento', 'fuerte', 'supera', 'sube',
        'ganancias', 'récord', 'mejora', 'positivo', 'optimista',
        'innovation', 'breakthrough', 'partnership', 'expansion',
        'revenue', 'earnings beat', 'raised guidance', 'dividend'
    ]

    palabras_negativas = [
        'sell', 'bearish', 'decline', 'weak', 'miss', 'drop',
        'fall', 'loss', 'downgrade', 'underperform', 'risk', 'concern',
        'venta', 'bajista', 'caída', 'débil', 'pierde', 'baja',
        'pérdidas', 'preocupación', 'negativo', 'pesimista',
        'lawsuit', 'investigation', 'recall', 'layoffs', 'debt',
        'inflation', 'recession', 'tariff', 'sanction', 'war'
    ]

    palabras_neutras = [
        'hold', 'neutral', 'stable', 'maintain', 'steady',
        'mantener', 'neutral', 'estable', 'sin cambios'
    ]

    score_total    = 0
    pos_encontradas = []
    neg_encontradas = []
    texto_completo  = ' '.join(textos).lower()

    for palabra in palabras_positivas:
        if palabra in texto_completo:
            score_total += 1
            pos_encontradas.append(palabra)

    for palabra in palabras_negativas:
        if palabra in texto_completo:
            score_total -= 1
            neg_encontradas.append(palabra)

    # Normalizar score entre -10 y 10
    score_norm = max(-10, min(10, score_total))

    if score_norm >= 4:
        descripcion = "😊 MUY POSITIVO"
        emoji       = "🟢"
    elif score_norm >= 2:
        descripcion = "🙂 POSITIVO"
        emoji       = "🟢"
    elif score_norm >= 0:
        descripcion = "😐 NEUTRAL/POSITIVO"
        emoji       = "🟡"
    elif score_norm >= -2:
        descripcion = "😕 NEUTRAL/NEGATIVO"
        emoji       = "🟡"
    elif score_norm >= -4:
        descripcion = "😟 NEGATIVO"
        emoji       = "🔴"
    else:
        descripcion = "😨 MUY NEGATIVO"
        emoji       = "🔴"

    return score_norm, descripcion, emoji, pos_encontradas[:5], neg_encontradas[:5]

# ============================================================
# ANÁLISIS GEOPOLÍTICO Y DE MERCADO
# ============================================================

def obtener_contexto_geopolitico(ticker, sector, pais="US"):
    """
    Obtiene contexto geopolítico y riesgos del mercado
    """
    print("  🌍 Buscando contexto geopolítico...")

    # ---- RIESGOS POR SECTOR ----
    riesgos_sector = {
        'Technology': [
            "⚠️ Regulación antimonopolio en EEUU y Europa",
            "⚠️ Restricciones exportación chips a China",
            "⚠️ Competencia con empresas chinas (Huawei, Alibaba)",
            "⚠️ Cambios en privacidad de datos (GDPR, CCPA)",
            "⚠️ Ciclos de inversión en IA y semiconductores"
        ],
        'Energy': [
            "⚠️ Volatilidad precio del petróleo (OPEP+)",
            "⚠️ Conflictos en Medio Oriente afectan suministro",
            "⚠️ Transición energética hacia renovables",
            "⚠️ Sanciones a Rusia afectan mercado global",
            "⚠️ Políticas climáticas y regulación ambiental"
        ],
        'Financial Services': [
            "⚠️ Cambios en tasas de interés de la Fed",
            "⚠️ Riesgo de recesión económica global",
            "⚠️ Regulación bancaria más estricta post-2008",
            "⚠️ Competencia de Fintech y criptomonedas",
            "⚠️ Exposición a deuda soberana de países emergentes"
        ],
        'Healthcare': [
            "⚠️ Aprobaciones FDA y regulación farmacéutica",
            "⚠️ Debate sobre precios de medicamentos en EEUU",
            "⚠️ Patentes y competencia de genéricos",
            "⚠️ Cambios en políticas de salud pública",
            "⚠️ Litigios y demandas colectivas"
        ],
        'Consumer Cyclical': [
            "⚠️ Inflación afecta poder adquisitivo",
            "⚠️ Tasas de interés altas reducen consumo",
            "⚠️ Disrupciones en cadena de suministro",
            "⚠️ Competencia del comercio electrónico chino",
            "⚠️ Cambios en hábitos de consumo post-COVID"
        ],
        'Communication Services': [
            "⚠️ Regulación de contenido y censura",
            "⚠️ Competencia por publicidad digital",
            "⚠️ Privacidad de datos y regulación",
            "⚠️ Saturación del mercado de streaming",
            "⚠️ Cambios en algoritmos afectan ingresos"
        ],
        'Industrials': [
            "⚠️ Aranceles y guerras comerciales",
            "⚠️ Costos de materias primas volátiles",
            "⚠️ Disrupciones en cadena de suministro global",
            "⚠️ Transición hacia manufactura verde",
            "⚠️ Competencia de manufactura asiática"
        ],
        'Consumer Defensive': [
            "⚠️ Inflación en costos de producción",
            "⚠️ Competencia de marcas privadas",
            "⚠️ Cambios en preferencias del consumidor",
            "⚠️ Regulación de alimentos y bebidas",
            "⚠️ Disrupciones climáticas afectan agricultura"
        ],
        'Real Estate': [
            "⚠️ Tasas de interés altas reducen demanda",
            "⚠️ Trabajo remoto afecta oficinas comerciales",
            "⚠️ Burbuja inmobiliaria en algunas regiones",
            "⚠️ Regulación de alquileres",
            "⚠️ Riesgo de recesión inmobiliaria"
        ],
        'Basic Materials': [
            "⚠️ Volatilidad en precios de commodities",
            "⚠️ Regulación ambiental más estricta",
            "⚠️ Dependencia de economía china",
            "⚠️ Conflictos geopolíticos afectan minería",
            "⚠️ Transición energética cambia demanda"
        ]
    }

    # ---- FACTORES GEOPOLÍTICOS GLOBALES ----
    factores_globales = [
        "🌍 Tensiones EEUU-China por Taiwan y comercio",
        "🌍 Conflicto Rusia-Ucrania afecta energía y granos",
        "🌍 Inflación global y políticas de bancos centrales",
        "🌍 Dolarización y fortaleza del USD",
        "🌍 Elecciones en EEUU 2024 generan incertidumbre",
        "🌍 Crisis de deuda en economías emergentes",
        "🌍 Transición energética y acuerdos climáticos"
    ]

    # ---- INDICADORES MACROECONÓMICOS ----
    macro_indicators = {
        "Fed Funds Rate":    "5.25-5.50% (Restrictivo)",
        "Inflación EEUU":    "~3.2% (Bajando gradualmente)",
        "GDP EEUU":          "~2.8% crecimiento anual",
        "Desempleo EEUU":    "~3.9% (Mercado laboral fuerte)",
        "VIX (Miedo)":       "Verificar en tiempo real",
        "DXY (Dólar)":       "Fuerte, presiona emergentes",
        "Rendimiento 10Y":   "~4.2% (Alto, presiona acciones)"
    }

    # Buscar noticias geopolíticas específicas
    noticias_geo = []
    queries_geo  = [
        f"{ticker} geopolitical risk 2024",
        f"{sector} sector market risk outlook",
        f"US economy stock market outlook 2024"
    ]

    for query in queries_geo:
        resultados = buscar_noticias(query, max_results=2)
        noticias_geo.extend(resultados)

    riesgos = riesgos_sector.get(sector, [
        "⚠️ Riesgo de mercado general",
        "⚠️ Volatilidad macroeconómica",
        "⚠️ Cambios regulatorios",
        "⚠️ Competencia del sector",
        "⚠️ Riesgo de tipo de cambio"
    ])

    return {
        'riesgos_sector':    riesgos,
        'factores_globales': factores_globales,
        'macro':             macro_indicators,
        'noticias_geo':      noticias_geo[:4]
    }

# ============================================================
# ANÁLISIS DE SENTIMIENTO DE INVERSORES
# ============================================================

def obtener_sentimiento_inversores(ticker, company_name):
    """
    Obtiene y analiza el sentimiento de los inversores
    """
    print("  💭 Analizando sentimiento de inversores...")

    todos_textos = []
    noticias_encontradas = []

    # Búsquedas múltiples
    queries = [
        f"{ticker} stock investor sentiment 2026",
        f"{ticker} stock forecast analyst opinion",
        f"{ticker} stock buy sell recommendation",
        f"{company_name} stock news today",
        f"{ticker} earnings outlook investors"
    ]

    for query in queries:
        # Buscar noticias
        noticias = buscar_noticias(query, max_results=3)
        for n in noticias:
            todos_textos.append(n['titulo'])
            noticias_encontradas.append(n)

        # Buscar texto
        textos = buscar_texto(query, max_results=2)
        todos_textos.extend(textos)

    # Analizar sentimiento
    if todos_textos:
        score, descripcion, emoji, pos, neg = analizar_sentimiento_texto(todos_textos)
    else:
        score, descripcion, emoji = 0, "😐 NEUTRAL (sin datos)", "🟡"
        pos, neg = [], []

    return {
        'score':       score,
        'descripcion': descripcion,
        'emoji':       emoji,
        'positivos':   pos,
        'negativos':   neg,
                'noticias':    noticias_encontradas[:6]
    }

# ============================================================
# FUNCIÓN PRINCIPAL DE ANÁLISIS COMPLETO
# ============================================================

def analizar_accion(ticker, periodo="6mo"):
    print(f"\n{'='*60}")
    print(f"📊 ANALIZANDO: {ticker.upper()}")
    print(f"⏰ Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}")

    try:
        # ---- OBTENER DATOS ----
        stock = yf.Ticker(ticker.upper())
        df    = stock.history(period=periodo)

        if df.empty:
            print(f"❌ No se encontraron datos para {ticker}")
            return

        try:
            info = stock.info
        except:
            info = {}

        company_name = info.get('longName', ticker.upper())
        sector       = info.get('sector', 'Technology')

        # ---- PRECIOS ----
        precio_actual   = df['Close'].iloc[-1]
        precio_anterior = df['Close'].iloc[-2]
        cambio          = ((precio_actual - precio_anterior) / precio_anterior) * 100

        print(f"\n💰 PRECIO ACTUAL:  ${precio_actual:.2f} ({cambio:+.2f}%)")
        print(f"🏢 EMPRESA:        {company_name}")
        print(f"🏭 SECTOR:         {sector}")
        print(f"🌍 INDUSTRIA:      {info.get('industry', 'N/A')}")
        print(f"\n🔄 Recopilando datos externos...")

        # ---- OBTENER CONTEXTO EXTERNO ----
        contexto_geo  = obtener_contexto_geopolitico(ticker, sector)
        sentimiento   = obtener_sentimiento_inversores(ticker, company_name)

        # ---- INDICADORES TÉCNICOS ----
        df['RSI']                                       = calcular_rsi(df['Close'])
        df['SMA20']                                     = df['Close'].rolling(window=20).mean()
        df['SMA50']                                     = df['Close'].rolling(window=50).mean()
        df['MACD'], df['MACD_Signal'], df['MACD_Hist'] = calcular_macd(df['Close'])
        df['BB_Upper'], df['BB_Middle'], df['BB_Lower'] = calcular_bollinger(df['Close'])
        df['ATR']                                       = calcular_atr(df['High'], df['Low'], df['Close'])

        # ---- VALORES ACTUALES ----
        rsi       = df['RSI'].iloc[-1]
        sma20     = df['SMA20'].iloc[-1]
        sma50     = df['SMA50'].iloc[-1]
        macd_val  = df['MACD'].iloc[-1]
        macd_sig  = df['MACD_Signal'].iloc[-1]
        bb_upper  = df['BB_Upper'].iloc[-1]
        bb_middle = df['BB_Middle'].iloc[-1]
        bb_lower  = df['BB_Lower'].iloc[-1]
        atr       = df['ATR'].iloc[-1]

        # ---- VALORES ANTERIORES ----
        macd_prev     = df['MACD'].iloc[-2]
        macd_sig_prev = df['MACD_Signal'].iloc[-2]
        sma20_prev    = df['SMA20'].iloc[-2]
        sma50_prev    = df['SMA50'].iloc[-2]

        # ---- SOPORTE Y RESISTENCIA ----
        soporte     = df['Low'].tail(60).quantile(0.05)
        resistencia = df['High'].tail(60).quantile(0.95)

        # ---- VOLATILIDAD ----
        retornos    = df['Close'].pct_change().dropna()
        volatilidad = retornos.std() * np.sqrt(252) * 100
        sharpe      = (retornos.mean() * 252) / (retornos.std() * np.sqrt(252))

        # ---- RETORNOS ----
        ret_1s  = ((df['Close'].iloc[-1] / df['Close'].iloc[-5])  - 1) * 100 if len(df) >= 5  else None
        ret_1m  = ((df['Close'].iloc[-1] / df['Close'].iloc[-21]) - 1) * 100 if len(df) >= 21 else None
        ret_3m  = ((df['Close'].iloc[-1] / df['Close'].iloc[-63]) - 1) * 100 if len(df) >= 63 else None
        ret_tot = ((df['Close'].iloc[-1] / df['Close'].iloc[0])   - 1) * 100

        # ---- ZONAS DE TRADING ----
        zona_compra_baja = soporte
        zona_compra_alta = soporte + (atr * 0.5)
        zona_venta_baja  = resistencia - (atr * 0.5)
        zona_venta_alta  = resistencia
        stop_loss_compra = soporte - (atr * 1.5)
        take_profit1     = precio_actual + (atr * 2)
        take_profit2     = precio_actual + (atr * 3)
        take_profit3     = resistencia
        stop_loss_venta  = resistencia + (atr * 1.5)
        target_bajista   = precio_actual - (atr * 3)
        riesgo_val       = precio_actual - stop_loss_compra
        beneficio_val    = take_profit1 - precio_actual
        ratio_rr         = beneficio_val / riesgo_val if riesgo_val > 0 else 0

        # ---- SEÑALES TÉCNICAS ----
        score_tecnico = 0
        señales       = []

        # RSI
        if rsi < 30:
            señales.append("🟢 RSI SOBREVENTA (<30)      → COMPRA FUERTE")
            score_tecnico += 2
        elif rsi < 40:
            señales.append("🟢 RSI Zona baja (30-40)     → COMPRA DÉBIL")
            score_tecnico += 1
        elif rsi > 70:
            señales.append("🔴 RSI SOBRECOMPRA (>70)     → VENTA FUERTE")
            score_tecnico -= 2
        elif rsi > 60:
            señales.append("🔴 RSI Zona alta (60-70)     → VENTA DÉBIL")
            score_tecnico -= 1
        else:
            señales.append(f"🟡 RSI Neutral ({rsi:.1f})        → Sin señal clara")

        # MACD
        if macd_val > macd_sig:
            señales.append("🟢 MACD sobre señal          → Tendencia ALCISTA")
            score_tecnico += 1.5
        else:
            señales.append("🔴 MACD bajo señal           → Tendencia BAJISTA")
            score_tecnico -= 1.5

        # Cruce MACD
        if macd_prev < macd_sig_prev and macd_val > macd_sig:
            señales.append("🟢 CRUCE ALCISTA MACD        → COMPRA FUERTE")
            score_tecnico += 2
        elif macd_prev > macd_sig_prev and macd_val < macd_sig:
            señales.append("🔴 CRUCE BAJISTA MACD        → VENTA FUERTE")
            score_tecnico -= 2

        # Medias Móviles
        if precio_actual > sma20:
            señales.append("🟢 Precio sobre SMA20        → Corto plazo ALCISTA")
            score_tecnico += 1
        else:
            señales.append("🔴 Precio bajo SMA20         → Corto plazo BAJISTA")
            score_tecnico -= 1

        if not np.isnan(sma50) and precio_actual > sma50:
            señales.append("🟢 Precio sobre SMA50        → Medio plazo ALCISTA")
            score_tecnico += 1
        elif not np.isnan(sma50):
            señales.append("🔴 Precio bajo SMA50         → Medio plazo BAJISTA")
            score_tecnico -= 1

        # Golden/Death Cross
        if not np.isnan(sma50_prev):
            if sma20_prev < sma50_prev and sma20 > sma50:
                señales.append("🟢 GOLDEN CROSS              → COMPRA MUY FUERTE")
                score_tecnico += 3
            elif sma20_prev > sma50_prev and sma20 < sma50:
                señales.append("🔴 DEATH CROSS               → VENTA MUY FUERTE")
                score_tecnico -= 3

        # Bollinger Bands
        if precio_actual <= bb_lower:
            señales.append("🟢 Precio en BB Inferior     → Posible REBOTE")
            score_tecnico += 1.5
        elif precio_actual >= bb_upper:
            señales.append("🔴 Precio en BB Superior     → Posible CORRECCIÓN")
            score_tecnico -= 1.5
        elif precio_actual > bb_middle:
            señales.append("🟡 Precio sobre BB Media     → Tendencia positiva")
            score_tecnico += 0.5
        else:
            señales.append("🟡 Precio bajo BB Media      → Tendencia negativa")
            score_tecnico -= 0.5

        # Soporte y Resistencia
        dist_soporte     = ((precio_actual - soporte) / soporte) * 100
        dist_resistencia = ((resistencia - precio_actual) / precio_actual) * 100

        if dist_soporte < 3:
            señales.append("🟢 Precio cerca del SOPORTE  → Zona de COMPRA")
            score_tecnico += 1
        if dist_resistencia < 3:
            señales.append("🔴 Precio cerca RESISTENCIA  → Zona de VENTA")
            score_tecnico -= 1

        # ---- SCORE DE SENTIMIENTO ----
        score_sentimiento = sentimiento['score']

        # ---- SCORE GEOPOLÍTICO ----
        # Penalizar según número de riesgos activos
        score_geo = 0
        num_riesgos = len(contexto_geo['riesgos_sector'])
        if num_riesgos >= 5:
            score_geo = -2
        elif num_riesgos >= 3:
            score_geo = -1
        else:
            score_geo = 0

        # ---- SCORE TOTAL COMBINADO ----
        score_total = (score_tecnico * 0.5) + (score_sentimiento * 0.3) + (score_geo * 0.2)

        # ---- RECOMENDACIÓN FINAL ----
        if score_total >= 4:
            recomendacion = "🟢 COMPRA FUERTE"
            accion        = "COMPRAR AHORA"
        elif score_total >= 2:
            recomendacion = "🟢 COMPRA"
            accion        = "COMPRAR"
        elif score_total >= 0.5:
            recomendacion = "🟡 COMPRA DÉBIL"
            accion        = "COMPRAR CON CAUTELA"
        elif score_total >= -0.5:
            recomendacion = "🟡 NEUTRAL"
            accion        = "MANTENER / ESPERAR"
        elif score_total >= -2:
            recomendacion = "🔴 VENTA DÉBIL"
            accion        = "REDUCIR POSICIÓN"
        elif score_total >= -4:
            recomendacion = "🔴 VENTA"
            accion        = "VENDER"
        else:
            recomendacion = "🔴 VENTA FUERTE"
            accion        = "VENDER AHORA"

        # ---- NIVEL DE RIESGO ----
        if volatilidad < 20:
            riesgo_nivel = "🟢 BAJO"
        elif volatilidad < 40:
            riesgo_nivel = "🟡 MODERADO"
        elif volatilidad < 60:
            riesgo_nivel = "🟠 ALTO"
        else:
            riesgo_nivel = "🔴 MUY ALTO"

        # ============================================================
        # IMPRIMIR REPORTE COMPLETO
        # ============================================================

        print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📈 RENDIMIENTO HISTÓRICO:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1 Semana:      {f"{ret_1s:+.2f}%" if ret_1s is not None else "N/A"}
  1 Mes:         {f"{ret_1m:+.2f}%" if ret_1m is not None else "N/A"}
  3 Meses:       {f"{ret_3m:+.2f}%" if ret_3m is not None else "N/A"}
  Total:         {ret_tot:+.2f}%

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚡ RIESGO Y VOLATILIDAD:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Volatilidad:   {volatilidad:.2f}%
  Sharpe Ratio:  {sharpe:.2f}
  Nivel Riesgo:  {riesgo_nivel}
  Beta:          {info.get('beta', 'N/A')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 INDICADORES TÉCNICOS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RSI (14):      {rsi:.2f}
  MACD:          {macd_val:.4f}
  MACD Señal:    {macd_sig:.4f}
  SMA 20:        ${sma20:.2f}
  SMA 50:        ${sma50:.2f}
  BB Superior:   ${bb_upper:.2f}
  BB Inferior:   ${bb_lower:.2f}
  ATR (14):      ${atr:.2f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📡 SEÑALES TÉCNICAS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")

        for s in señales:
            print(f"  {s}")

        print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💭 SENTIMIENTO DE INVERSORES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Sentimiento:   {sentimiento['emoji']} {sentimiento['descripcion']}
  Score:         {sentimiento['score']}/10""")

        if sentimiento['positivos']:
            print(f"  Factores +:    {', '.join(sentimiento['positivos'])}")
        if sentimiento['negativos']:
            print(f"  Factores -:    {', '.join(sentimiento['negativos'])}")

        if sentimiento['noticias']:
            print(f"\n  📰 NOTICIAS RECIENTES:")
            for n in sentimiento['noticias'][:5]:
                if n.get('titulo'):
                    fecha  = n.get('fecha', '')[:10] if n.get('fecha') else ''
                    fuente = n.get('fuente', '')
                    print(f"  • [{fecha}] {n['titulo'][:70]}...")
                    if fuente:
                        print(f"    Fuente: {fuente}")

        print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🌍 CONTEXTO GEOPOLÍTICO Y RIESGOS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  🏭 RIESGOS ESPECÍFICOS DEL SECTOR ({sector}):""")

        for r in contexto_geo['riesgos_sector']:
            print(f"  {r}")

        print(f"""
  🌐 FACTORES GEOPOLÍTICOS GLOBALES:""")

        for f in contexto_geo['factores_globales']:
            print(f"  {f}")

        print(f"""
  📊 INDICADORES MACROECONÓMICOS:""")

        for k, v in contexto_geo['macro'].items():
            print(f"  • {k:<22} {v}")

        if contexto_geo['noticias_geo']:
            print(f"\n  📰 NOTICIAS GEOPOLÍTICAS:")
            for n in contexto_geo['noticias_geo']:
                if n.get('titulo'):
                    fecha  = n.get('fecha', '')[:10] if n.get('fecha') else ''
                    fuente = n.get('fuente', '')
                    print(f"  • [{fecha}] {n['titulo'][:70]}...")
                    if fuente:
                        print(f"    Fuente: {fuente}")

        print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 NIVELES CLAVE DE PRECIO:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Precio Actual: ${precio_actual:.2f}
  Soporte:       ${soporte:.2f}       (Dist: {dist_soporte:.1f}%)
  Resistencia:   ${resistencia:.2f}   (Dist: {dist_resistencia:.1f}%)
  52W High:      ${info.get('fiftyTwoWeekHigh', 'N/A')}
  52W Low:       ${info.get('fiftyTwoWeekLow',  'N/A')}
  Target Analis: ${info.get('targetMeanPrice',  'N/A')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💚 ESTRATEGIA DE COMPRA:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Zona Compra:   ${zona_compra_baja:.2f} - ${zona_compra_alta:.2f}
  Stop Loss:     ${stop_loss_compra:.2f}  ({((stop_loss_compra - precio_actual)/precio_actual*100):.1f}%)
  Take Profit 1: ${take_profit1:.2f}      ({((take_profit1 - precio_actual)/precio_actual*100):.1f}%)
  Take Profit 2: ${take_profit2:.2f}      ({((take_profit2 - precio_actual)/precio_actual*100):.1f}%)
  Take Profit 3: ${take_profit3:.2f}      ({((take_profit3 - precio_actual)/precio_actual*100):.1f}%)
  Ratio R/B:     {ratio_rr:.2f}x

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❤️  ESTRATEGIA DE VENTA:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Zona Venta:    ${zona_venta_baja:.2f} - ${zona_venta_alta:.2f}
  Stop Loss:     ${stop_loss_venta:.2f}   ({((stop_loss_venta - precio_actual)/precio_actual*100):.1f}%)
  Target:        ${target_bajista:.2f}    ({((target_bajista - precio_actual)/precio_actual*100):.1f}%)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💼 INFORMACIÓN FUNDAMENTAL:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  P/E Ratio:     {info.get('trailingPE',   'N/A')}
  P/E Forward:   {info.get('forwardPE',    'N/A')}
  Market Cap:    ${info.get('marketCap', 0)/1e9:.2f}B
  EPS:           {info.get('trailingEps',  'N/A')}
  ROE:           {f"{info.get('returnOnEquity',0)*100:.1f}%" if info.get('returnOnEquity') else 'N/A'}
  Margen Neto:   {f"{info.get('profitMargins',0)*100:.1f}%"  if info.get('profitMargins')  else 'N/A'}
  Dividendo:     {f"{info.get('dividendYield',0)*100:.2f}%"  if info.get('dividendYield')  else 'N/A'}
  Recom Analis:  {info.get('recommendationKey', 'N/A').upper()}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 SCORES DE ANÁLISIS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Score Técnico:      {score_tecnico:+.1f}  (peso 50%)
  Score Sentimiento:  {score_sentimiento:+.1f}  (peso 30%)
  Score Geopolítico:  {score_geo:+.1f}  (peso 20%)
  ─────────────────────────────
  SCORE TOTAL:        {score_total:+.1f} / 10

╔══════════════════════════════════════════════════════════╗
║                 🏆 RECOMENDACIÓN FINAL                  ║
╠══════════════════════════════════════════════════════════╣
║  Decisión:    {recomendacion:<43} ║
║  Acción:      {accion:<43} ║
║  Riesgo:      {riesgo_nivel:<43} ║
╚══════════════════════════════════════════════════════════╝
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️  DISCLAIMER: Solo informativo. No es asesoramiento
    financiero. Invierte bajo tu propio riesgo.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        print("💡 Verifica que el ticker sea correcto")


# ============================================================
# MENÚ INTERACTIVO
# ============================================================

def menu():
    print("""
╔══════════════════════════════════════════════════════════╗
║       📈 ANALIZADOR DE ACCIONES MR. ORJUELA v2.0.       ║
║                                                         ║
║  ✅ Análisis Técnico Completo                           ║
║  ✅ Sentimiento de Inversores en Tiempo Real            ║
║  ✅ Contexto Geopolítico y Riesgos                      ║
║  ✅ Indicadores Macroeconómicos                         ║
║  ✅ Noticias Recientes del Mercado                      ║
╚══════════════════════════════════════════════════════════╝
    """)

    while True:
        print("\n¿Qué deseas hacer?")
        print("1️⃣  Analizar una acción")
        print("2️⃣  Analizar varias acciones")
        print("3️⃣  Ver lista de tickers populares")
        print("4️⃣  Salir")

        opcion = input("\nElige opción (1-4): ").strip()

        if opcion == "1":
            ticker  = input("\nIngresa el ticker (ej: AAPL): ").strip().upper()
            periodo = input("Período (1mo/3mo/6mo/1y/2y) [Enter=6mo]: ").strip()
            if not periodo:
                periodo = "6mo"
            analizar_accion(ticker, periodo)

        elif opcion == "2":
            tickers = input("\nIngresa tickers separados por coma (ej: AAPL,TSLA,MSFT): ")
            periodo = input("Período [Enter=6mo]: ").strip()
            if not periodo:
                periodo = "6mo"
            lista = [t.strip().upper() for t in tickers.split(",")]
            for t in lista:
                analizar_accion(t, periodo)

        elif opcion == "3":
            print("""
╔══════════════════════════════════════════════╗
║         📊 TICKERS POPULARES                ║
╠══════════════════════════════════════════════╣
║  TECNOLOGÍA:                                ║
║  AAPL  = Apple                              ║
║  MSFT  = Microsoft                          ║
║  GOOGL = Google                             ║
║  META  = Meta                               ║
║  NVDA  = Nvidia                             ║
║  AMD   = AMD                                ║
║  TSLA  = Tesla                              ║
║  AMZN  = Amazon                             ║
╠══════════════════════════════════════════════╣
║  FINANZAS:                                  ║
║  JPM   = JP Morgan                          ║
║  BAC   = Bank of America                    ║
║  GS    = Goldman Sachs                      ║
╠══════════════════════════════════════════════╣
║  SALUD:                                     ║
║  JNJ   = Johnson & Johnson                  ║
║  PFE   = Pfizer                             ║
║  UNH   = UnitedHealth
   ABBV  = Abbvie Inc.                        ║
╠══════════════════════════════════════════════╣
║  ENERGÍA:                                   ║
║  XOM   = ExxonMobil                         ║
║  CVX   = Chevron                            ║
╠══════════════════════════════════════════════╣
║  ETFs ÍNDICES:                              ║
║  SPY   = S&P 500                            ║
║  QQQ   = Nasdaq 100                         ║
║  DIA   = Dow Jones                          ║
╠══════════════════════════════════════════════╣
║  CRYPTO ETF:                                ║
║  IBIT  = Bitcoin ETF BlackRock              ║
║  ETHA  = Ethereum ETF BlackRock             ║
╚══════════════════════════════════════════════╝
            """)
            ticker  = input("Elige un ticker: ").strip().upper()
            periodo = input("Período [Enter=6mo]: ").strip()
            if not periodo:
                periodo = "6mo"
            analizar_accion(ticker, periodo)

        elif opcion == "4":
            print("\n👋 ¡Hasta luego! Que tus inversiones sean exitosas! 📈\n")
            break

        else:
            print("❌ Opción inválida, elige entre 1-4")
 #============================================================
            # CELDA 3: EJECUTAR
# ============================================================

# Opción A: Menú interactivo
menu()

# Opción B: Directo sin menú
# analizar_accion("AAPL")
# analizar_accion("TSLA", "1y")
# analizar_accion("NVDA", "3mo")


⚠️ duckduckgo_search no disponible, usando datos alternativos

╔══════════════════════════════════════════════════════════╗
║       📈 ANALIZADOR DE ACCIONES MR. ORJUELA v2.0.       ║
║                                                         ║
║  ✅ Análisis Técnico Completo                           ║
║  ✅ Sentimiento de Inversores en Tiempo Real            ║
║  ✅ Contexto Geopolítico y Riesgos                      ║
║  ✅ Indicadores Macroeconómicos                         ║
║  ✅ Noticias Recientes del Mercado                      ║
╚══════════════════════════════════════════════════════════╝
    

¿Qué deseas hacer?
1️⃣  Analizar una acción
2️⃣  Analizar varias acciones
3️⃣  Ver lista de tickers populares
4️⃣  Salir

Elige opción (1-4): 1

Ingresa el ticker (ej: AAPL): GOOGL
Período (1mo/3mo/6mo/1y/2y) [Enter=6mo]: 1y

📊 ANALIZANDO: GOOGL
⏰ Fecha: 2026-04-20 20:18:15

💰 PRECIO ACTUAL:  $337.42 (-1.25%)
🏢 EMPRESA:        Alphabet Inc.
🏭 SECTOR:         Communication Services
🌍 INDUSTRIA

KeyboardInterrupt: Interrupted by user